In [1]:
from __future__ import annotations

import json
from datetime import datetime, timezone
from uuid import uuid4

import pandas as pd
import vertexai
from google import genai
from google.api_core.exceptions import Conflict, NotFound
from google.cloud import bigquery
from google.cloud import storage
from vertexai.language_models import TextEmbeddingModel

In [2]:
PROJECT_ID = "leafy-guide-497515-m4"
LOCATION = "us-central1"

BUCKET_NAME = "leafy-guide-497515-m4-vector-assets"

DATASET_ID = "sre_incident_runbook_rag"
DOCUMENT_TABLE_ID = "sre_documents"
CHUNK_TABLE_ID = "sre_document_chunks"
INCIDENT_TABLE_ID = "sre_incidents"
POSTMORTEM_TABLE_ID = "sre_postmortems"

PLANNING_MODEL = "gemini-2.5-pro"
TEXT_MODEL = "gemini-2.5-flash"
TEXT_EMBEDDING_MODEL = "text-embedding-005"

CHUNK_SIZE_CHARS = 1200
CHUNK_OVERLAP_CHARS = 200
MAX_EMBEDDING_TEXT_CHARS = 3000

GCS_SUMMARY_PREFIX = "sre-incident-runbook-rag/summaries"

print("Configuration loaded.")
print("Project:", PROJECT_ID)
print("Location:", LOCATION)
print("Bucket:", BUCKET_NAME)
print("Planning model:", PLANNING_MODEL)
print("Text model:", TEXT_MODEL)
print("Embedding model:", TEXT_EMBEDDING_MODEL)

Configuration loaded.
Project: leafy-guide-497515-m4
Location: us-central1
Bucket: leafy-guide-497515-m4-vector-assets
Planning model: gemini-2.5-pro
Text model: gemini-2.5-flash
Embedding model: text-embedding-005


In [3]:
vertexai.init(
    project=PROJECT_ID,
    location=LOCATION,
)

google_vertex_client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION,
)

storage_client = storage.Client(project=PROJECT_ID)
bucket = storage_client.bucket(BUCKET_NAME)

bigquery_client = bigquery.Client(project=PROJECT_ID)

text_embedding_model = TextEmbeddingModel.from_pretrained(
    TEXT_EMBEDDING_MODEL
)

print("Clients created.")
print("Bucket exists:", bucket.exists())
print("BigQuery project:", bigquery_client.project)
print("Text embedding model loaded.")

/home/lipov/projects/Python_practice_sessions_05/.venv/lib/python3.13/site-packages/vertexai/_model_garden/_model_garden_models.py:278: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


Clients created.
Bucket exists: True
BigQuery project: leafy-guide-497515-m4
Text embedding model loaded.


In [4]:
dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
dataset_ref.location = "EU"

try:
    dataset = bigquery_client.create_dataset(dataset_ref)
    print("Created dataset:", dataset.full_dataset_id)
except Conflict:
    dataset = bigquery_client.get_dataset(dataset_ref)
    print("Dataset already exists:", dataset.full_dataset_id)

document_table_ref = f"{PROJECT_ID}.{DATASET_ID}.{DOCUMENT_TABLE_ID}"
chunk_table_ref = f"{PROJECT_ID}.{DATASET_ID}.{CHUNK_TABLE_ID}"
incident_table_ref = f"{PROJECT_ID}.{DATASET_ID}.{INCIDENT_TABLE_ID}"
postmortem_table_ref = f"{PROJECT_ID}.{DATASET_ID}.{POSTMORTEM_TABLE_ID}"

print("Document table:", document_table_ref)
print("Chunk table:", chunk_table_ref)
print("Incident table:", incident_table_ref)
print("Postmortem table:", postmortem_table_ref)

Created dataset: leafy-guide-497515-m4:sre_incident_runbook_rag
Document table: leafy-guide-497515-m4.sre_incident_runbook_rag.sre_documents
Chunk table: leafy-guide-497515-m4.sre_incident_runbook_rag.sre_document_chunks
Incident table: leafy-guide-497515-m4.sre_incident_runbook_rag.sre_incidents
Postmortem table: leafy-guide-497515-m4.sre_incident_runbook_rag.sre_postmortems


In [5]:
def ensure_bigquery_table(
    table_ref: str,
    schema: list[bigquery.SchemaField],
) -> bigquery.Table:
    try:
        table = bigquery_client.get_table(table_ref)
        print("Table already exists:", table.full_table_id)
        return table

    except NotFound:
        print("Table does not exist. Creating:", table_ref)

        table = bigquery.Table(
            table_ref,
            schema=schema,
        )

        created_table = bigquery_client.create_table(table)
        print("Created table:", created_table.full_table_id)

        return created_table

In [6]:
document_schema = [
    bigquery.SchemaField("document_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("document_number", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("document_type", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("system_name", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("service_name", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("title", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("source_type", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("content", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("summary", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("created_at", "TIMESTAMP", mode="REQUIRED"),
]

chunk_schema = [
    bigquery.SchemaField("chunk_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("document_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("document_number", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("chunk_number", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("global_chunk_number", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("document_type", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("system_name", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("service_name", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("title", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("source_type", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("chunk_text", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("chunk_char_count", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("embedding_model", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("embedding", "FLOAT64", mode="REPEATED"),
    bigquery.SchemaField("created_at", "TIMESTAMP", mode="REQUIRED"),
]

incident_schema = [
    bigquery.SchemaField("incident_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("incident_number", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("incident_title", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("severity", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("primary_service", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("impacted_services", "STRING", mode="REPEATED"),
    bigquery.SchemaField("symptoms", "STRING", mode="REPEATED"),
    bigquery.SchemaField("signal_text", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("expected_root_cause", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("created_at", "TIMESTAMP", mode="REQUIRED"),
]

postmortem_schema = [
    bigquery.SchemaField("postmortem_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("technician_question", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("answer", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("used_chunk_ids", "STRING", mode="REPEATED"),
    bigquery.SchemaField("used_document_titles", "STRING", mode="REPEATED"),
    bigquery.SchemaField("created_at", "TIMESTAMP", mode="REQUIRED"),
]

document_table = ensure_bigquery_table(
    document_table_ref,
    document_schema,
)

chunk_table = ensure_bigquery_table(
    chunk_table_ref,
    chunk_schema,
)

incident_table = ensure_bigquery_table(
    incident_table_ref,
    incident_schema,
)

postmortem_table = ensure_bigquery_table(
    postmortem_table_ref,
    postmortem_schema,
)

Table does not exist. Creating: leafy-guide-497515-m4.sre_incident_runbook_rag.sre_documents
Created table: leafy-guide-497515-m4:sre_incident_runbook_rag.sre_documents
Table does not exist. Creating: leafy-guide-497515-m4.sre_incident_runbook_rag.sre_document_chunks
Created table: leafy-guide-497515-m4:sre_incident_runbook_rag.sre_document_chunks
Table does not exist. Creating: leafy-guide-497515-m4.sre_incident_runbook_rag.sre_incidents
Created table: leafy-guide-497515-m4:sre_incident_runbook_rag.sre_incidents
Table does not exist. Creating: leafy-guide-497515-m4.sre_incident_runbook_rag.sre_postmortems
Created table: leafy-guide-497515-m4:sre_incident_runbook_rag.sre_postmortems


In [7]:
def gcs_uri_from_blob_name(
    bucket_name: str,
    blob_name: str,
) -> str:
    return f"gs://{bucket_name}/{blob_name}"


def upload_text_to_gcs(
    text: str,
    *,
    blob_name: str,
    content_type: str = "application/json",
) -> str:
    blob = bucket.blob(blob_name)

    blob.upload_from_string(
        text,
        content_type=content_type,
    )

    return gcs_uri_from_blob_name(
        bucket_name=BUCKET_NAME,
        blob_name=blob_name,
    )

In [8]:
def extract_json_object(text: str) -> dict:
    cleaned = text.strip()

    if cleaned.startswith("```json"):
        cleaned = cleaned.removeprefix("```json").strip()

    if cleaned.startswith("```"):
        cleaned = cleaned.removeprefix("```").strip()

    if cleaned.endswith("```"):
        cleaned = cleaned.removesuffix("```").strip()

    start = cleaned.find("{")
    end = cleaned.rfind("}")

    if start == -1 or end == -1:
        raise ValueError(f"No JSON object found in text: {text[:500]}")

    return json.loads(cleaned[start:end + 1])

In [9]:
sre_brief = """
Create a synthetic SRE / DevOps knowledge base for a cloud-native ecommerce platform.

The platform contains:
- checkout-service
- payment-service
- inventory-service
- recommendation-service
- api-gateway
- postgres-order-db
- redis-cache
- kafka-order-events

The knowledge base should support:
- incident triage
- root cause analysis
- semantic search over runbook chunks
- on-call assistant answers
- postmortem drafting
- deployment risk analysis
- alert interpretation

The data should feel like realistic internal runbooks, alert documents, deployment notes, and incident signals.
"""

print(sre_brief)


Create a synthetic SRE / DevOps knowledge base for a cloud-native ecommerce platform.

The platform contains:
- checkout-service
- payment-service
- inventory-service
- recommendation-service
- api-gateway
- postgres-order-db
- redis-cache
- kafka-order-events

The knowledge base should support:
- incident triage
- root cause analysis
- semantic search over runbook chunks
- on-call assistant answers
- postmortem drafting
- deployment risk analysis
- alert interpretation

The data should feel like realistic internal runbooks, alert documents, deployment notes, and incident signals.



In [10]:
runbook_specs = [
    {
        "document_number": 1,
        "document_type": "runbook",
        "system_name": "ecommerce_platform",
        "service_name": "checkout-service",
        "title": "Checkout Service Latency Runbook",
        "source_type": "generated_runbook",
        "focus": "latency spikes, cart validation delays, downstream timeouts, dependency checks, rollback criteria",
    },
    {
        "document_number": 2,
        "document_type": "runbook",
        "system_name": "ecommerce_platform",
        "service_name": "payment-service",
        "title": "Payment Service Error Rate Runbook",
        "source_type": "generated_runbook",
        "focus": "payment gateway timeout, retry storms, idempotency keys, provider degradation, circuit breaker status",
    },
    {
        "document_number": 3,
        "document_type": "runbook",
        "system_name": "ecommerce_platform",
        "service_name": "postgres-order-db",
        "title": "Order Database Connection Exhaustion Runbook",
        "source_type": "generated_runbook",
        "focus": "connection pool saturation, slow queries, transaction locks, p95 latency, emergency pool tuning",
    },
    {
        "document_number": 4,
        "document_type": "runbook",
        "system_name": "ecommerce_platform",
        "service_name": "redis-cache",
        "title": "Redis Cache Stampede Runbook",
        "source_type": "generated_runbook",
        "focus": "cache misses, hot keys, TTL alignment, thundering herd, fallback protection",
    },
    {
        "document_number": 5,
        "document_type": "runbook",
        "system_name": "ecommerce_platform",
        "service_name": "kafka-order-events",
        "title": "Kafka Order Events Lag Runbook",
        "source_type": "generated_runbook",
        "focus": "consumer lag, partition imbalance, poison messages, retry topics, dead letter queue analysis",
    },
]

pd.DataFrame(runbook_specs)

,document_number,document_type,system_name,service_name,title,source_type,focus
0,1,runbook,ecommerce_platform,checkout-service,Checkout Service Latency Runbook,generated_runbook,"latency spikes, cart validation delays, downst..."
1,2,runbook,ecommerce_platform,payment-service,Payment Service Error Rate Runbook,generated_runbook,"payment gateway timeout, retry storms, idempot..."
2,3,runbook,ecommerce_platform,postgres-order-db,Order Database Connection Exhaustion Runbook,generated_runbook,"connection pool saturation, slow queries, tran..."
3,4,runbook,ecommerce_platform,redis-cache,Redis Cache Stampede Runbook,generated_runbook,"cache misses, hot keys, TTL alignment, thunder..."
4,5,runbook,ecommerce_platform,kafka-order-events,Kafka Order Events Lag Runbook,generated_runbook,"consumer lag, partition imbalance, poison mess..."


In [11]:
runbook_specs = [
    {
        "document_number": 1,
        "document_type": "runbook",
        "system_name": "ecommerce_platform",
        "service_name": "checkout-service",
        "title": "Checkout Service Latency Runbook",
        "source_type": "generated_runbook",
        "focus": "latency spikes, cart validation delays, downstream timeouts, dependency checks, rollback criteria",
    },
    {
        "document_number": 2,
        "document_type": "runbook",
        "system_name": "ecommerce_platform",
        "service_name": "payment-service",
        "title": "Payment Service Error Rate Runbook",
        "source_type": "generated_runbook",
        "focus": "payment gateway timeout, retry storms, idempotency keys, provider degradation, circuit breaker status",
    },
    {
        "document_number": 3,
        "document_type": "runbook",
        "system_name": "ecommerce_platform",
        "service_name": "postgres-order-db",
        "title": "Order Database Connection Exhaustion Runbook",
        "source_type": "generated_runbook",
        "focus": "connection pool saturation, slow queries, transaction locks, p95 latency, emergency pool tuning",
    },
    {
        "document_number": 4,
        "document_type": "runbook",
        "system_name": "ecommerce_platform",
        "service_name": "redis-cache",
        "title": "Redis Cache Stampede Runbook",
        "source_type": "generated_runbook",
        "focus": "cache misses, hot keys, TTL alignment, thundering herd, fallback protection",
    },
    {
        "document_number": 5,
        "document_type": "runbook",
        "system_name": "ecommerce_platform",
        "service_name": "kafka-order-events",
        "title": "Kafka Order Events Lag Runbook",
        "source_type": "generated_runbook",
        "focus": "consumer lag, partition imbalance, poison messages, retry topics, dead letter queue analysis",
    },
]

pd.DataFrame(runbook_specs)

,document_number,document_type,system_name,service_name,title,source_type,focus
0,1,runbook,ecommerce_platform,checkout-service,Checkout Service Latency Runbook,generated_runbook,"latency spikes, cart validation delays, downst..."
1,2,runbook,ecommerce_platform,payment-service,Payment Service Error Rate Runbook,generated_runbook,"payment gateway timeout, retry storms, idempot..."
2,3,runbook,ecommerce_platform,postgres-order-db,Order Database Connection Exhaustion Runbook,generated_runbook,"connection pool saturation, slow queries, tran..."
3,4,runbook,ecommerce_platform,redis-cache,Redis Cache Stampede Runbook,generated_runbook,"cache misses, hot keys, TTL alignment, thunder..."
4,5,runbook,ecommerce_platform,kafka-order-events,Kafka Order Events Lag Runbook,generated_runbook,"consumer lag, partition imbalance, poison mess..."


In [12]:
incident_specs = [
    {
        "incident_number": 1,
        "incident_title": "Checkout latency spike after cart validation release",
        "severity": "SEV2",
        "primary_service": "checkout-service",
        "impacted_services": ["checkout-service", "inventory-service", "postgres-order-db"],
        "symptoms": [
            "checkout p95 latency increased from 350ms to 4200ms",
            "cart validation timeout errors increased",
            "database connection pool near saturation",
            "new deployment completed 12 minutes before alert",
        ],
        "expected_root_cause": "cart validation release increased synchronous inventory and database calls",
    },
    {
        "incident_number": 2,
        "incident_title": "Payment errors during provider degradation",
        "severity": "SEV1",
        "primary_service": "payment-service",
        "impacted_services": ["payment-service", "api-gateway"],
        "symptoms": [
            "payment authorization error rate above 18%",
            "provider timeout errors visible in logs",
            "retry volume increased sharply",
            "circuit breaker remained half-open for too long",
        ],
        "expected_root_cause": "external payment provider degradation combined with aggressive retry policy",
    },
    {
        "incident_number": 3,
        "incident_title": "Recommendation cache stampede during traffic surge",
        "severity": "SEV2",
        "primary_service": "recommendation-service",
        "impacted_services": ["recommendation-service", "redis-cache"],
        "symptoms": [
            "redis hit rate dropped below 55%",
            "recommendation-service CPU increased",
            "hot product keys expired at the same time",
            "fallback response rate increased",
        ],
        "expected_root_cause": "aligned TTLs caused hot key expiration and thundering herd behavior",
    },
    {
        "incident_number": 4,
        "incident_title": "Kafka order events lag after malformed message",
        "severity": "SEV2",
        "primary_service": "kafka-order-events",
        "impacted_services": ["kafka-order-events", "inventory-service", "checkout-service"],
        "symptoms": [
            "consumer lag exceeded 250000 messages",
            "one partition stopped progressing",
            "poison message retries repeated",
            "inventory reservation events delayed",
        ],
        "expected_root_cause": "poison message blocked one consumer partition and retry handling was not isolated",
    },
]

pd.DataFrame(incident_specs)

,incident_number,incident_title,severity,primary_service,impacted_services,symptoms,expected_root_cause
0,1,Checkout latency spike after cart validation r...,SEV2,checkout-service,"[checkout-service, inventory-service, postgres...",[checkout p95 latency increased from 350ms to ...,cart validation release increased synchronous ...
1,2,Payment errors during provider degradation,SEV1,payment-service,"[payment-service, api-gateway]","[payment authorization error rate above 18%, p...",external payment provider degradation combined...
2,3,Recommendation cache stampede during traffic s...,SEV2,recommendation-service,"[recommendation-service, redis-cache]","[redis hit rate dropped below 55%, recommendat...",aligned TTLs caused hot key expiration and thu...
3,4,Kafka order events lag after malformed message,SEV2,kafka-order-events,"[kafka-order-events, inventory-service, checko...","[consumer lag exceeded 250000 messages, one pa...",poison message blocked one consumer partition ...


In [14]:
def generate_runbook_document(spec: dict) -> str:
    prompt = f"""
You are a senior SRE writing an internal runbook.

SRE brief:
{sre_brief}

Runbook metadata:
{json.dumps(spec, indent=2, ensure_ascii=False)}

Write a realistic internal runbook in English with these sections:
1. Service overview
2. Common alerts
3. Symptoms and first checks
4. Dependency checks
5. Useful metrics
6. Log patterns
7. Step-by-step triage
8. Rollback or mitigation options
9. Escalation rules
10. Post-incident notes

Requirements:
- Make it realistic but synthetic.
- Include concrete metric names, log patterns, and operational steps.
- Include enough content to be split into multiple semantic chunks.
- Do not use markdown tables.
- Do not mention that this is AI-generated.
"""

    response = google_vertex_client.models.generate_content(
        model=PLANNING_MODEL,
        contents=prompt,
    )

    return response.text.strip()


runbook_documents = []

for spec in runbook_specs:
    print("=" * 100)
    print("Generating runbook:", spec["title"])

    content = generate_runbook_document(spec)

    document = {
        "document_id": str(uuid4()),
        "document_number": spec["document_number"],
        "document_type": spec["document_type"],
        "system_name": spec["system_name"],
        "service_name": spec["service_name"],
        "title": spec["title"],
        "source_type": spec["source_type"],
        "content": content,
        "summary": content[:700],
        "created_at": datetime.now(timezone.utc).isoformat(),
    }

    runbook_documents.append(document)

    print("Characters:", len(content))
    print(content[:500])

Generating runbook: Checkout Service Latency Runbook
Characters: 12164
{
  "document_number": 1,
  "document_type": "runbook",
  "system_name": "ecommerce_platform",
  "service_name": "checkout-service",
  "title": "Checkout Service Latency Runbook",
  "source_type": "generated_runbook",
  "focus": "latency spikes, cart validation delays, downstream timeouts, dependency checks, rollback criteria"
}

### 1. Service Overview

The `checkout-service` is a stateful Go microservice that orchestrates the customer checkout process. It is a critical, revenue-impacting servi
Generating runbook: Payment Service Error Rate Runbook
Characters: 12537
{
  "document_number": 2,
  "document_type": "runbook",
  "system_name": "ecommerce_platform",
  "service_name": "payment-service",
  "title": "Payment Service Error Rate Runbook",
  "source_type": "generated_runbook",
  "focus": "payment gateway timeout, retry storms, idempotency keys, provider degradation, circuit breaker status"
}

### 1. Service Ove

In [15]:
def generate_incident_signal_text(spec: dict) -> str:
    prompt = f"""
You are generating a synthetic SRE incident signal document.

SRE brief:
{sre_brief}

Incident metadata:
{json.dumps(spec, indent=2, ensure_ascii=False)}

Create an internal incident signal document in English.

Include:
1. Alert summary
2. Timeline of events
3. Metrics snapshot
4. Representative log lines
5. Deployment notes
6. Initial hypotheses
7. Known mitigations attempted
8. Handoff notes for the next on-call engineer

Requirements:
- Make it realistic but synthetic.
- Use concrete metric names and log-like snippets.
- Do not use markdown tables.
- Do not mention that it is AI-generated.
"""

    response = google_vertex_client.models.generate_content(
        model=PLANNING_MODEL,
        contents=prompt,
    )

    return response.text.strip()


incident_documents = []
incident_rows = []

for spec in incident_specs:
    print("=" * 100)
    print("Generating incident signal:", spec["incident_title"])

    signal_text = generate_incident_signal_text(spec)

    incident_id = str(uuid4())

    incident_rows.append(
        {
            "incident_id": incident_id,
            "incident_number": spec["incident_number"],
            "incident_title": spec["incident_title"],
            "severity": spec["severity"],
            "primary_service": spec["primary_service"],
            "impacted_services": spec["impacted_services"],
            "symptoms": spec["symptoms"],
            "signal_text": signal_text,
            "expected_root_cause": spec["expected_root_cause"],
            "created_at": datetime.now(timezone.utc).isoformat(),
        }
    )

    document = {
        "document_id": str(uuid4()),
        "document_number": len(runbook_documents) + spec["incident_number"],
        "document_type": "incident_signal",
        "system_name": "ecommerce_platform",
        "service_name": spec["primary_service"],
        "title": f"Incident Signal - {spec['incident_title']}",
        "source_type": "generated_incident_signal",
        "content": signal_text,
        "summary": signal_text[:700],
        "created_at": datetime.now(timezone.utc).isoformat(),
    }

    incident_documents.append(document)

    print("Characters:", len(signal_text))
    print(signal_text[:500])

Generating incident signal: Checkout latency spike after cart validation release
Characters: 5155
**INCIDENT SIGNAL DOCUMENT: INC-001**

**1. Alert summary**

- **Alert Name:** High p95 Latency on checkout-service
- **Time Fired:** 2023-10-27 14:15 UTC
- **Severity:** SEV2
- **Summary:** The `checkout-service` is experiencing a severe latency degradation on its primary endpoints. P95 latency has increased by over 10x, leading to a significant increase in user-facing errors and cart abandonment signals. The incident began shortly after a new version of the service was deployed.

**2. Timeline
Generating incident signal: Payment errors during provider degradation
Characters: 6106
### SRE Incident Signal Document

**Incident Number:** 2
**Alert Source:** P1-PaymentAuthErrorRateHigh
**Service:** payment-service
**Signal Time:** 2023-10-26 13:45 UTC

---

### 1. Alert Summary
A SEV1 has been declared for `payment-service`. The P1 alert `PaymentAuthErrorRateHigh` fired at 13:45 UTC, indicati

In [16]:
all_documents = [
    *runbook_documents,
    *incident_documents,
]

print("Runbook documents:", len(runbook_documents))
print("Incident documents:", len(incident_documents))
print("Total documents:", len(all_documents))

documents_df = pd.DataFrame(
    [
        {
            "document_number": document["document_number"],
            "document_type": document["document_type"],
            "service_name": document["service_name"],
            "title": document["title"],
            "content_length": len(document["content"]),
        }
        for document in all_documents
    ]
)

documents_df

Runbook documents: 5
Incident documents: 4
Total documents: 9


,document_number,document_type,service_name,title,content_length
0,1,runbook,checkout-service,Checkout Service Latency Runbook,12164
1,2,runbook,payment-service,Payment Service Error Rate Runbook,12537
2,3,runbook,postgres-order-db,Order Database Connection Exhaustion Runbook,10998
3,4,runbook,redis-cache,Redis Cache Stampede Runbook,10377
4,5,runbook,kafka-order-events,Kafka Order Events Lag Runbook,12009
5,6,incident_signal,checkout-service,Incident Signal - Checkout latency spike after...,5155
6,7,incident_signal,payment-service,Incident Signal - Payment errors during provid...,6106
7,8,incident_signal,recommendation-service,Incident Signal - Recommendation cache stamped...,6840
8,9,incident_signal,kafka-order-events,Incident Signal - Kafka order events lag after...,8384


In [17]:
def normalize_text(text: str) -> str:
    return " ".join(text.split())


def split_text_into_chunks(
    text: str,
    *,
    chunk_size_chars: int = CHUNK_SIZE_CHARS,
    chunk_overlap_chars: int = CHUNK_OVERLAP_CHARS,
) -> list[str]:
    clean_text = normalize_text(text)

    if len(clean_text) <= chunk_size_chars:
        return [clean_text]

    chunks = []
    start = 0

    while start < len(clean_text):
        end = min(start + chunk_size_chars, len(clean_text))

        if end < len(clean_text):
            sentence_boundary = clean_text.rfind(". ", start, end)

            if sentence_boundary > start + int(chunk_size_chars * 0.6):
                end = sentence_boundary + 1

        chunk = clean_text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        if end >= len(clean_text):
            break

        start = max(0, end - chunk_overlap_chars)

    return chunks

In [18]:
chunk_rows_without_embeddings = []
global_chunk_number = 1

for document in all_documents:
    chunks = split_text_into_chunks(document["content"])

    print("=" * 100)
    print("Document:", document["title"])
    print("Chunks:", len(chunks))

    for chunk_index, chunk_text in enumerate(chunks, start=1):
        chunk_rows_without_embeddings.append(
            {
                "chunk_id": str(uuid4()),
                "document_id": document["document_id"],
                "document_number": document["document_number"],
                "chunk_number": chunk_index,
                "global_chunk_number": global_chunk_number,
                "document_type": document["document_type"],
                "system_name": document["system_name"],
                "service_name": document["service_name"],
                "title": document["title"],
                "source_type": document["source_type"],
                "chunk_text": chunk_text,
                "chunk_char_count": len(chunk_text),
                "embedding_model": TEXT_EMBEDDING_MODEL,
                "created_at": datetime.now(timezone.utc).isoformat(),
            }
        )

        global_chunk_number += 1

print("Total chunks:", len(chunk_rows_without_embeddings))

chunks_preview_df = pd.DataFrame(
    [
        {
            "global_chunk_number": row["global_chunk_number"],
            "service_name": row["service_name"],
            "title": row["title"],
            "chunk_char_count": row["chunk_char_count"],
            "preview": row["chunk_text"][:180],
        }
        for row in chunk_rows_without_embeddings
    ]
)

chunks_preview_df.head(10)

Document: Checkout Service Latency Runbook
Chunks: 13
Document: Payment Service Error Rate Runbook
Chunks: 14
Document: Order Database Connection Exhaustion Runbook
Chunks: 12
Document: Redis Cache Stampede Runbook
Chunks: 11
Document: Kafka Order Events Lag Runbook
Chunks: 13
Document: Incident Signal - Checkout latency spike after cart validation release
Chunks: 6
Document: Incident Signal - Payment errors during provider degradation
Chunks: 7
Document: Incident Signal - Recommendation cache stampede during traffic surge
Chunks: 7
Document: Incident Signal - Kafka order events lag after malformed message
Chunks: 9
Total chunks: 92


,global_chunk_number,service_name,title,chunk_char_count,preview
0,1,checkout-service,Checkout Service Latency Runbook,1194,"{ ""document_number"": 1, ""document_type"": ""runb..."
1,2,checkout-service,Checkout Service Latency Runbook,1159,*: Ingress for all user-facing traffic. - **in...
2,3,checkout-service,Checkout Service Latency Runbook,1164,koutServiceErrorRateSpike`** - **Description**...
3,4,checkout-service,Checkout Service Latency Runbook,1197,"s and First Checks **User Impact**: - The ""Pla..."
4,5,checkout-service,Checkout Service Latency Runbook,1112,d is the top suspect. ### 4. Dependency Checks...
5,6,checkout-service,Checkout Service Latency Runbook,1160,"dashboard for high latency, high CPU, or memor..."
6,7,checkout-service,Checkout Service Latency Runbook,1160,or a lock. - **DB Connection Pool Usage**: `db...
7,8,checkout-service,Checkout Service Latency Runbook,1163,"e DB connection from pool""` - This is a clear ..."
8,9,checkout-service,Checkout Service Latency Runbook,1017,ard filters. 3. **Check for Deployments**: Thi...
9,10,checkout-service,Checkout Service Latency Runbook,1146,db_pool_in_use`: Is it maxed out? This points ...


In [19]:
def shorten_text_for_embedding(
    text: str,
    *,
    max_chars: int = MAX_EMBEDDING_TEXT_CHARS,
) -> str:
    clean_text = normalize_text(text)

    if len(clean_text) <= max_chars:
        return clean_text

    return clean_text[:max_chars].rsplit(" ", 1)[0]


def get_text_embedding(text: str) -> list[float]:
    safe_text = shorten_text_for_embedding(text)

    embeddings = text_embedding_model.get_embeddings(
        [safe_text]
    )

    return list(embeddings[0].values)

In [20]:
chunk_rows = []

for row in chunk_rows_without_embeddings:
    print(
        "Embedding chunk:",
        row["global_chunk_number"],
        "|",
        row["service_name"],
        "|",
        row["title"],
    )

    embedding = get_text_embedding(row["chunk_text"])

    chunk_rows.append(
        {
            **row,
            "embedding": embedding,
        }
    )

    print("Embedding dim:", len(embedding))
    print("Chunk chars:", row["chunk_char_count"])
    print("-" * 100)

print("Chunk rows with embeddings:", len(chunk_rows))

Embedding chunk: 1 | checkout-service | Checkout Service Latency Runbook
Embedding dim: 768
Chunk chars: 1194
----------------------------------------------------------------------------------------------------
Embedding chunk: 2 | checkout-service | Checkout Service Latency Runbook
Embedding dim: 768
Chunk chars: 1159
----------------------------------------------------------------------------------------------------
Embedding chunk: 3 | checkout-service | Checkout Service Latency Runbook
Embedding dim: 768
Chunk chars: 1164
----------------------------------------------------------------------------------------------------
Embedding chunk: 4 | checkout-service | Checkout Service Latency Runbook
Embedding dim: 768
Chunk chars: 1197
----------------------------------------------------------------------------------------------------
Embedding chunk: 5 | checkout-service | Checkout Service Latency Runbook
Embedding dim: 768
Chunk chars: 1112
----------------------------------------------

In [21]:
document_rows = []

for document in all_documents:
    document_rows.append(
        {
            "document_id": document["document_id"],
            "document_number": document["document_number"],
            "document_type": document["document_type"],
            "system_name": document["system_name"],
            "service_name": document["service_name"],
            "title": document["title"],
            "source_type": document["source_type"],
            "content": document["content"],
            "summary": document["summary"],
            "created_at": document["created_at"],
        }
    )

bigquery_client.query(
    f"TRUNCATE TABLE `{document_table_ref}`",
    location=dataset.location,
).result()

bigquery_client.query(
    f"TRUNCATE TABLE `{chunk_table_ref}`",
    location=dataset.location,
).result()

bigquery_client.query(
    f"TRUNCATE TABLE `{incident_table_ref}`",
    location=dataset.location,
).result()

document_errors = bigquery_client.insert_rows_json(
    document_table_ref,
    document_rows,
)

chunk_errors = bigquery_client.insert_rows_json(
    chunk_table_ref,
    chunk_rows,
)

incident_errors = bigquery_client.insert_rows_json(
    incident_table_ref,
    incident_rows,
)

if document_errors:
    print("Document insert errors:")
    print(document_errors)
else:
    print("Inserted document rows:", len(document_rows))

if chunk_errors:
    print("Chunk insert errors:")
    print(chunk_errors)
else:
    print("Inserted chunk rows:", len(chunk_rows))

if incident_errors:
    print("Incident insert errors:")
    print(incident_errors)
else:
    print("Inserted incident rows:", len(incident_rows))

Inserted document rows: 9
Inserted chunk rows: 92
Inserted incident rows: 4


In [22]:
sql = f"""
SELECT
  (SELECT COUNT(*) FROM `{document_table_ref}`) AS document_count,
  (SELECT COUNT(*) FROM `{chunk_table_ref}`) AS chunk_count,
  (SELECT COUNT(*) FROM `{incident_table_ref}`) AS incident_count,
  (
    SELECT ARRAY_LENGTH(embedding)
    FROM `{chunk_table_ref}`
    LIMIT 1
  ) AS embedding_length
"""

result = list(
    bigquery_client.query(
        sql,
        location=dataset.location,
    )
)[0]

print("Document count:", result.document_count)
print("Chunk count:", result.chunk_count)
print("Incident count:", result.incident_count)
print("Embedding length:", result.embedding_length)

Document count: 9
Chunk count: 92
Incident count: 4
Embedding length: 768


In [23]:
def search_sre_chunks(
    query: str,
    *,
    top_k: int = 8,
    service_name: str | None = None,
) -> list[dict]:
    query_embedding = get_text_embedding(query)

    service_filter_sql = ""

    if service_name is not None:
        service_filter_sql = "WHERE service_name = @service_name"

    sql = f"""
    SELECT
      base.chunk_id,
      base.document_id,
      base.document_number,
      base.chunk_number,
      base.global_chunk_number,
      base.document_type,
      base.system_name,
      base.service_name,
      base.title,
      base.source_type,
      base.chunk_text,
      distance
    FROM VECTOR_SEARCH(
      (
        SELECT *
        FROM `{chunk_table_ref}`
        {service_filter_sql}
      ),
      'embedding',
      (SELECT @query_embedding AS embedding),
      top_k => @top_k,
      distance_type => 'COSINE'
    )
    ORDER BY distance ASC
    """

    query_parameters = [
        bigquery.ArrayQueryParameter(
            "query_embedding",
            "FLOAT64",
            query_embedding,
        ),
        bigquery.ScalarQueryParameter(
            "top_k",
            "INT64",
            top_k,
        ),
    ]

    if service_name is not None:
        query_parameters.append(
            bigquery.ScalarQueryParameter(
                "service_name",
                "STRING",
                service_name,
            )
        )

    job_config = bigquery.QueryJobConfig(
        query_parameters=query_parameters,
    )

    query_job = bigquery_client.query(
        sql,
        job_config=job_config,
        location=dataset.location,
    )

    return [
        {
            "chunk_id": row.chunk_id,
            "document_id": row.document_id,
            "document_number": row.document_number,
            "chunk_number": row.chunk_number,
            "global_chunk_number": row.global_chunk_number,
            "document_type": row.document_type,
            "system_name": row.system_name,
            "service_name": row.service_name,
            "title": row.title,
            "source_type": row.source_type,
            "chunk_text": row.chunk_text,
            "distance": row.distance,
        }
        for row in query_job
    ]

In [24]:
query = """
Checkout latency increased after a release. Cart validation is timing out,
database connections are high, and inventory calls look slower than normal.
What should the on-call engineer check first?
"""

search_results = search_sre_chunks(
    query,
    top_k=8,
)

print("Query:")
print(query)

print("\nSearch results:", len(search_results))

for result in search_results:
    print("=" * 100)
    print("Distance:", round(result["distance"], 4))
    print("Service:", result["service_name"])
    print("Document:", result["title"])
    print("Document type:", result["document_type"])
    print("Global chunk:", result["global_chunk_number"])
    print(result["chunk_text"][:900])

Query:

Checkout latency increased after a release. Cart validation is timing out,
database connections are high, and inventory calls look slower than normal.
What should the on-call engineer check first?


Search results: 8
Distance: 0.2121
Service: checkout-service
Document: Checkout Service Latency Runbook
Document type: runbook
Global chunk: 4
s and First Checks **User Impact**: - The "Place Order" button on the checkout page spins indefinitely. - Users receive a generic "Something went wrong, please try again" error page after a long wait. - Carts fail to load or update correctly at the final review step. **First Checks for On-Call**: 1. **Dashboard Review**: Immediately open the "Checkout Service - Primary" Grafana dashboard. 2. **Latency Graph**: Look at the "P99/P95 Latency: /v1/checkout/place-order" panel. Is this a sudden spike, a gradual climb, or a plateau at a high level? 3. **Pod Health**: Check the health of the service pods. Are they all in a `Running` state? Any recent

In [25]:
filtered_results = search_sre_chunks(
    query,
    top_k=5,
    service_name="checkout-service",
)

print("Filtered search results:", len(filtered_results))

for result in filtered_results:
    print("=" * 100)
    print("Distance:", round(result["distance"], 4))
    print("Service:", result["service_name"])
    print("Document:", result["title"])
    print("Global chunk:", result["global_chunk_number"])
    print(result["chunk_text"][:700])

Filtered search results: 5
Distance: 0.2121
Service: checkout-service
Document: Checkout Service Latency Runbook
Global chunk: 4
s and First Checks **User Impact**: - The "Place Order" button on the checkout page spins indefinitely. - Users receive a generic "Something went wrong, please try again" error page after a long wait. - Carts fail to load or update correctly at the final review step. **First Checks for On-Call**: 1. **Dashboard Review**: Immediately open the "Checkout Service - Primary" Grafana dashboard. 2. **Latency Graph**: Look at the "P99/P95 Latency: /v1/checkout/place-order" panel. Is this a sudden spike, a gradual climb, or a plateau at a high level? 3. **Pod Health**: Check the health of the service pods. Are they all in a `Running` state? Any recent restarts? `kubectl get pods -n checkout -l app=ch
Distance: 0.22
Service: checkout-service
Document: Checkout Service Latency Runbook
Global chunk: 5
d is the top suspect. ### 4. Dependency Checks High latency in `checko

In [26]:
def build_rag_context_from_chunks(results: list[dict]) -> str:
    parts = []

    for index, result in enumerate(results, start=1):
        parts.append(
            "\n".join(
                [
                    f"[SOURCE {index}]",
                    f"Document title: {result['title']}",
                    f"Document type: {result['document_type']}",
                    f"Service name: {result['service_name']}",
                    f"Source type: {result['source_type']}",
                    f"Global chunk number: {result['global_chunk_number']}",
                    f"Chunk number in document: {result['chunk_number']}",
                    f"Distance: {result['distance']:.4f}",
                    f"Chunk text: {result['chunk_text']}",
                ]
            )
        )

    return "\n\n---\n\n".join(parts)

In [27]:
def build_rag_context_from_chunks(results: list[dict]) -> str:
    parts = []

    for index, result in enumerate(results, start=1):
        parts.append(
            "\n".join(
                [
                    f"[SOURCE {index}]",
                    f"Document title: {result['title']}",
                    f"Document type: {result['document_type']}",
                    f"Service name: {result['service_name']}",
                    f"Source type: {result['source_type']}",
                    f"Global chunk number: {result['global_chunk_number']}",
                    f"Chunk number in document: {result['chunk_number']}",
                    f"Distance: {result['distance']:.4f}",
                    f"Chunk text: {result['chunk_text']}",
                ]
            )
        )

    return "\n\n---\n\n".join(parts)

In [28]:
def build_rag_context_from_chunks(results: list[dict]) -> str:
    parts = []

    for index, result in enumerate(results, start=1):
        parts.append(
            "\n".join(
                [
                    f"[SOURCE {index}]",
                    f"Document title: {result['title']}",
                    f"Document type: {result['document_type']}",
                    f"Service name: {result['service_name']}",
                    f"Source type: {result['source_type']}",
                    f"Global chunk number: {result['global_chunk_number']}",
                    f"Chunk number in document: {result['chunk_number']}",
                    f"Distance: {result['distance']:.4f}",
                    f"Chunk text: {result['chunk_text']}",
                ]
            )
        )

    return "\n\n---\n\n".join(parts)

In [31]:
def answer_sre_question(
    oncall_question: str,
    *,
    top_k: int = 8,
    service_name: str | None = None,
) -> dict:
    results = search_sre_chunks(
        oncall_question,
        top_k=top_k,
        service_name=service_name,
    )

    context = build_rag_context_from_chunks(results)

    prompt = f"""
You are a senior SRE assistant.

Answer the on-call engineer using only the retrieved chunks.

On-call question:
{oncall_question}

Retrieved chunks:
{context}

Return a practical incident response answer with:
1. Most likely root cause
2. Immediate checks
3. Metrics to inspect
4. Logs to inspect
5. Mitigation steps
6. Rollback criteria
7. Escalation criteria
8. Sources used, referencing SOURCE numbers

Rules:
- Do not invent information outside the retrieved chunks.
- If evidence is insufficient, say what is missing.
- Be operational and concise.
"""

    response = google_vertex_client.models.generate_content(
        model=TEXT_MODEL,
        contents=prompt,
    )

    return {
        "oncall_question": oncall_question,
        "answer": response.text,
        "search_results": results,
        "rag_context": context,
    }

In [32]:
oncall_question = """
We have a SEV2 where checkout p95 jumped above 4 seconds shortly after a deployment.
Cart validation is timing out, inventory calls are slower, and the order database pool is close to saturation.
What should I check first, and when should I rollback?
"""

sre_result = answer_sre_question(
    oncall_question,
    top_k=8,
)

print("ON-CALL QUESTION:")
print(sre_result["oncall_question"])

print("\nSRE RAG ANSWER:")
print(sre_result["answer"])

ON-CALL QUESTION:

We have a SEV2 where checkout p95 jumped above 4 seconds shortly after a deployment.
Cart validation is timing out, inventory calls are slower, and the order database pool is close to saturation.
What should I check first, and when should I rollback?


SRE RAG ANSWER:
Here's how to approach this SEV2 incident based on the provided information:

**1. Most likely root cause:**
The most likely root cause is a recent deployment of the `checkout-service`, specifically `v1.22.5`, which introduced a new real-time cart validation feature. This feature has caused:
*   A sharp increase in calls from `checkout-service` to `inventory-service`.
*   A corresponding rise in active connections to `postgres-order-db`.
*   Latency spikes and resource exhaustion (DB connections) due to synchronous, blocking calls for each item in the cart, leading to long-running locking transactions on the database.
(SOURCE 1, 3, 4, 5, 6)

**2. Immediate checks:**
1.  **Check for Deployments:** This i

In [33]:
def generate_postmortem_draft(rag_result: dict) -> dict:
    used_chunk_ids = [
        result["chunk_id"]
        for result in rag_result["search_results"]
    ]

    used_document_titles = sorted(
        {
            result["title"]
            for result in rag_result["search_results"]
        }
    )

    prompt = f"""
You are an SRE lead writing a postmortem draft.

Use the on-call question, retrieved context, and RAG answer below.

On-call question:
{rag_result["oncall_question"]}

Retrieved context:
{rag_result["rag_context"]}

RAG answer:
{rag_result["answer"]}

Write a postmortem draft with:
1. Executive summary
2. Customer impact
3. Timeline
4. Root cause hypothesis
5. Detection
6. Resolution
7. What went well
8. What went wrong
9. Follow-up action items
10. Evidence sources

Rules:
- Stay grounded in the retrieved chunks.
- If something is unknown, mark it as unknown.
- Do not invent exact timestamps unless present in the context.
"""

    response = google_vertex_client.models.generate_content(
        model=PLANNING_MODEL,
        contents=prompt,
    )

    return {
        "postmortem_id": str(uuid4()),
        "technician_question": rag_result["oncall_question"],
        "answer": response.text,
        "used_chunk_ids": used_chunk_ids,
        "used_document_titles": used_document_titles,
        "created_at": datetime.now(timezone.utc).isoformat(),
    }


postmortem = generate_postmortem_draft(sre_result)

print(postmortem["answer"])

Here is a postmortem draft based on the provided information.

***

## **Postmortem Draft: SEV2 - Checkout Service Latency Spike**

### 1. Executive Summary

On 2023-10-27, starting at 14:12 UTC, the checkout service experienced a severe degradation, with p95 latency increasing over 10x to more than 4 seconds. This resulted in a significant increase in checkout failures and user-facing timeout errors (HTTP 504). The incident was triggered by the deployment of `checkout-service:v1.22.5`, which introduced a new "real-time cart validation" feature. This feature's design caused an 8x increase in calls to the inventory service and exhausted the order database's connection pool by creating long-running locking transactions. The on-call engineer correctly identified the deployment as the trigger and resolved the incident by rolling back to the previous stable version, `v1.22.4`.

### 2. Customer Impact

*   **Impact Start:** 2023-10-27 14:12 UTC
*   **Impact End:** [Time Unknown - when rollba

In [34]:
bigquery_client.query(
    f"TRUNCATE TABLE `{postmortem_table_ref}`",
    location=dataset.location,
).result()

postmortem_errors = bigquery_client.insert_rows_json(
    postmortem_table_ref,
    [postmortem],
)

if postmortem_errors:
    print("Postmortem insert errors:")
    print(postmortem_errors)
else:
    print("Inserted postmortem rows: 1")

Inserted postmortem rows: 1


In [35]:
test_questions = [
    "Payment authorization failures increased and the provider is timing out. What should we check?",
    "Redis hit rate dropped and recommendation CPU went high after hot keys expired.",
    "Kafka consumer lag is growing and one partition stopped moving. What should the on-call do?",
    "Order database has high connection usage and slow checkout writes.",
]

for question in test_questions:
    print("\n" + "=" * 100)
    print("QUESTION:")
    print(question)

    results = search_sre_chunks(
        question,
        top_k=4,
    )

    for result in results:
        print(
            "distance:",
            round(result["distance"], 4),
            "| service:",
            result["service_name"],
            "| doc:",
            result["title"],
            "| chunk:",
            result["global_chunk_number"],
        )


QUESTION:
Payment authorization failures increased and the provider is timing out. What should we check?
distance: 0.2606 | service: payment-service | doc: Incident Signal - Payment errors during provider degradation | chunk: 70
distance: 0.2803 | service: payment-service | doc: Incident Signal - Payment errors during provider degradation | chunk: 72
distance: 0.2866 | service: payment-service | doc: Incident Signal - Payment errors during provider degradation | chunk: 76
distance: 0.2868 | service: payment-service | doc: Incident Signal - Payment errors during provider degradation | chunk: 71

QUESTION:
Redis hit rate dropped and recommendation CPU went high after hot keys expired.
distance: 0.2143 | service: recommendation-service | doc: Incident Signal - Recommendation cache stampede during traffic surge | chunk: 78
distance: 0.2478 | service: recommendation-service | doc: Incident Signal - Recommendation cache stampede during traffic surge | chunk: 79
distance: 0.2612 | service: r

In [36]:
sql = f"""
SELECT
  service_name,
  document_type,
  source_type,
  COUNT(*) AS chunk_count,
  ROUND(AVG(chunk_char_count), 2) AS avg_chunk_chars,
  MIN(chunk_char_count) AS min_chunk_chars,
  MAX(chunk_char_count) AS max_chunk_chars
FROM `{chunk_table_ref}`
GROUP BY
  service_name,
  document_type,
  source_type
ORDER BY
  service_name,
  chunk_count DESC
"""

chunk_analytics_df = bigquery_client.query(
    sql,
    location=dataset.location,
).to_dataframe()

chunk_analytics_df

/home/lipov/projects/Python_practice_sessions_05/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,service_name,document_type,source_type,chunk_count,avg_chunk_chars,min_chunk_chars,max_chunk_chars
0,checkout-service,runbook,generated_runbook,13,1108.38,733,1197
1,checkout-service,incident_signal,generated_incident_signal,6,1021.17,858,1155
2,kafka-order-events,runbook,generated_runbook,13,1080.85,423,1193
3,kafka-order-events,incident_signal,generated_incident_signal,9,1098.11,1001,1198
4,payment-service,runbook,generated_runbook,14,1066.79,467,1193
5,payment-service,incident_signal,generated_incident_signal,7,1028.57,730,1195
6,postgres-order-db,runbook,generated_runbook,12,1081.58,749,1200
7,recommendation-service,incident_signal,generated_incident_signal,7,1132.43,1048,1191
8,redis-cache,runbook,generated_runbook,11,1106.64,920,1185


In [37]:
sql = f"""
SELECT
  incident_number,
  incident_title,
  severity,
  primary_service,
  impacted_services,
  symptoms,
  expected_root_cause
FROM `{incident_table_ref}`
ORDER BY incident_number
"""

incident_overview_df = bigquery_client.query(
    sql,
    location=dataset.location,
).to_dataframe()

incident_overview_df

/home/lipov/projects/Python_practice_sessions_05/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,incident_number,incident_title,severity,primary_service,impacted_services,symptoms,expected_root_cause
0,1,Checkout latency spike after cart validation r...,SEV2,checkout-service,"[checkout-service, inventory-service, postgres...",[checkout p95 latency increased from 350ms to ...,cart validation release increased synchronous ...
1,2,Payment errors during provider degradation,SEV1,payment-service,"[payment-service, api-gateway]","[payment authorization error rate above 18%, p...",external payment provider degradation combined...
2,3,Recommendation cache stampede during traffic s...,SEV2,recommendation-service,"[recommendation-service, redis-cache]","[redis hit rate dropped below 55%, recommendat...",aligned TTLs caused hot key expiration and thu...
3,4,Kafka order events lag after malformed message,SEV2,kafka-order-events,"[kafka-order-events, inventory-service, checko...","[consumer lag exceeded 250000 messages, one pa...",poison message blocked one consumer partition ...


In [38]:
notebook_summary = {
    "project_id": PROJECT_ID,
    "location": LOCATION,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "notebook": "61_w_google_cloud_sre_incident_runbook_rag.ipynb",
    "models": {
        "planning_model": PLANNING_MODEL,
        "text_model": TEXT_MODEL,
        "text_embedding_model": TEXT_EMBEDDING_MODEL,
    },
    "bigquery": {
        "dataset": DATASET_ID,
        "document_table": document_table_ref,
        "chunk_table": chunk_table_ref,
        "incident_table": incident_table_ref,
        "postmortem_table": postmortem_table_ref,
    },
    "sre_brief": sre_brief,
    "document_count": len(all_documents),
    "chunk_count": len(chunk_rows),
    "incident_count": len(incident_rows),
    "sre_result": sre_result,
    "postmortem": postmortem,
    "chunk_analytics": chunk_analytics_df.to_dict(orient="records"),
}

summary_json = json.dumps(
    notebook_summary,
    indent=2,
    ensure_ascii=False,
    default=str,
)

timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
blob_name = f"{GCS_SUMMARY_PREFIX}/summary_{timestamp}.json"

summary_gcs_uri = upload_text_to_gcs(
    summary_json,
    blob_name=blob_name,
)

print("Notebook summary saved to:")
print(summary_gcs_uri)

Notebook summary saved to:
gs://leafy-guide-497515-m4-vector-assets/sre-incident-runbook-rag/summaries/summary_20260701_142152.json


In [39]:
print("SRE Incident Runbook RAG notebook completed.")
print("=" * 100)

print("Documents:", len(all_documents))
print("Chunks:", len(chunk_rows))
print("Incidents:", len(incident_rows))

print("\nBigQuery tables:")
print("-", document_table_ref)
print("-", chunk_table_ref)
print("-", incident_table_ref)
print("-", postmortem_table_ref)

print("\nChunk analytics:")

for row in chunk_analytics_df.to_dict(orient="records"):
    print("=" * 100)
    print("Service:", row["service_name"])
    print("Document type:", row["document_type"])
    print("Source type:", row["source_type"])
    print("Chunk count:", row["chunk_count"])
    print("Average chunk chars:", row["avg_chunk_chars"])

print("\nExample SRE RAG answer:")
print(sre_result["answer"][:1400])

print("\nSummary GCS URI:")
print(summary_gcs_uri)

SRE Incident Runbook RAG notebook completed.
Documents: 9
Chunks: 92
Incidents: 4

BigQuery tables:
- leafy-guide-497515-m4.sre_incident_runbook_rag.sre_documents
- leafy-guide-497515-m4.sre_incident_runbook_rag.sre_document_chunks
- leafy-guide-497515-m4.sre_incident_runbook_rag.sre_incidents
- leafy-guide-497515-m4.sre_incident_runbook_rag.sre_postmortems

Chunk analytics:
Service: checkout-service
Document type: runbook
Source type: generated_runbook
Chunk count: 13
Average chunk chars: 1108.38
Service: checkout-service
Document type: incident_signal
Source type: generated_incident_signal
Chunk count: 6
Average chunk chars: 1021.17
Service: kafka-order-events
Document type: runbook
Source type: generated_runbook
Chunk count: 13
Average chunk chars: 1080.85
Service: kafka-order-events
Document type: incident_signal
Source type: generated_incident_signal
Chunk count: 9
Average chunk chars: 1098.11
Service: payment-service
Document type: runbook
Source type: generated_runbook
Chunk cou